# 02 — Collaborative Filtering

**Group 9 contributors**

- ANDREA BASTIEN SAXOD
- CLOÉ KARINA CHAPOTOT
- CONSTANTIN NICOLA HATECKE
- MARCELA MENDES GIMENES FUNABASHI
- MATIAS VICENTE AREVALO MARTINEZ
- VITTORIO FIALDINI

## Objective

Learn team-specific preferences from historical team-player minutes. The non-personalized baseline gives every club the same ranking; collaborative filtering personalises it so that Arsenal and Brentford see different candidate lists.

## Inputs

- `players_scored` from notebook 01, filtered to seasons 2018-19 through 2022-23.
- Interaction weight: `log1p(minutes_played)` per team-player pair, summed across seasons in the training window.

## Method summary

1. Build a sparse team-by-player interaction matrix. Rows are clubs, columns are players, values are summed `log1p(minutes_played)` across training seasons.
2. Factorise the matrix with `TruncatedSVD(k=16)` to get team factors and item factors.
3. Restrict candidates to the previous season's pool and exclude players who were already at the target club that season.
4. Score candidates with the dot product between the target team's factor vector and each candidate's item vector.

## Algorithmic note, read before the shortlists

`TruncatedSVD` on a log-minutes matrix is an algorithmic compromise, not the textbook choice. The proper approach for implicit feedback (Hu, Koren, Volinsky 2008) is Alternating Least Squares with confidence weighting, which explicitly distinguishes a missing interaction ("we never signed this player", low confidence zero) from an observed one ("we used them heavily", high confidence positive). SVD instead treats unobserved entries as reconstruction targets at zero and minimises error across both observed and unobserved cells, which blurs that distinction. In practice the latent directions that SVD recovers on a sparse log-minutes matrix are still useful for ranking similar-style players, but the interpretation of the factors is weaker than under ALS. We document this openly rather than paper over it, and note that switching to `implicit.als` would be the natural next step.

## Key assumptions

- Clubs who play a player a lot effectively endorse that player's style and fit.
- A player's minutes in the training window capture something about the type of team that rates them.
- k=16 is a reasonable but not optimised choice. The k-stability cell below shows how much the top-10 shifts if k is halved or doubled.

## Outputs

- A trained CF model with team and item factor matrices.
- Example shortlists for Arsenal (MF), Liverpool (DF) and Brentford (FW) targeting the 2023-24 window.
- A k-ablation table showing top-10 Jaccard overlap for k in {8, 16, 32}.


In [1]:
import warnings, json
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def locate_project_root():
    """Find a root directory that contains the expected data files.
    Supports Colab layout (/content/Recommendation_engine_group), a data/ folder
    with real_uploaded and synthetic subfolders, or a flat upload folder.
    """
    candidates = [
        Path('/content/Recommendation_engine_group'),
        Path('/content'),
        Path('/mnt/user-data/uploads'),
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/home/claude'),
    ]
    for base in candidates:
        if not base.exists():
            continue
        has_structured = (base / 'data').exists() and (
            (base / 'data' / 'real').exists()
            or (base / 'data' / 'real_uploaded').exists()
            or (base / 'data' / 'synthetic').exists()
        )
        has_flat = (base / 'players_master_with_market_values.csv').exists() or (
            base / 'players_master_synthetic_augmented.csv'
        ).exists()
        if has_structured or has_flat:
            return base
    raise FileNotFoundError('Could not locate project root with the expected data files.')

ROOT = locate_project_root()
print('Using project root:', ROOT)

def read_first_existing(paths, **kwargs):
    last_err = None
    for p in paths:
        p = Path(p)
        if p.exists():
            try:
                return pd.read_csv(p, **kwargs)
            except Exception as e:
                last_err = e
    if last_err is not None:
        raise last_err
    raise FileNotFoundError(f'None of the candidate paths exist: {paths}')

def load_real_tables(root=ROOT):
    players = read_first_existing([
        root / 'data' / 'real_uploaded' / 'players_master_with_market_values.csv',
        root / 'data' / 'real' / 'players_master_with_market_values.csv.gz',
        root / 'players_master_with_market_values.csv',
    ], low_memory=False)
    teams = read_first_existing([
        root / 'data' / 'real_uploaded' / 'teams_master_with_market_values.csv',
        root / 'data' / 'real' / 'teams_master_with_market_values.csv.gz',
        root / 'teams_master_with_market_values.csv',
    ], low_memory=False)
    return players, teams

def load_synthetic_tables(root=ROOT):
    players_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'players_master_synthetic_augmented.csv',
        root / 'players_master_synthetic_augmented.csv',
    ], low_memory=False)
    teams_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'teams_master_synthetic_augmented.csv',
        root / 'teams_master_synthetic_augmented.csv',
    ], low_memory=False)
    return players_aug, teams_aug

TRAIN_SEASONS = ['2018-19','2019-20','2020-21','2021-22']
VAL_SEASON = '2022-23'
TEST_SEASON = '2023-24'
PREV_SEASON = {'2019-20':'2018-19','2020-21':'2019-20','2021-22':'2020-21','2022-23':'2021-22','2023-24':'2022-23'}

PER90_MAP = {
    'misc__Performance__Int':'interceptions_p90',
    'misc__Performance__TklW':'tackles_won_p90',
    'misc__Performance__Fls':'fouls_committed_p90',
    'misc__Performance__Fld':'fouls_won_p90',
    'misc__Performance__Crs':'crosses_p90',
    'misc__Performance__Off':'offsides_p90',
}

REAL_SCORE_FEATURES = [
    'standard__Per 90 Minutes__G-PK',
    'standard__Per 90 Minutes__Ast',
    'shooting__Standard__Sh/90',
    'shooting__Standard__SoT/90',
    'shooting__Standard__G/Sh',
    'shooting__Standard__SoT%',
    'interceptions_p90',
    'tackles_won_p90',
    'crosses_p90',
    'fouls_won_p90',
    'offsides_p90',
    'playing_time__Team Success__+/-90',
    'playing_time__Team Success__On-Off',
    'playing_time__Playing Time__Min%',
]

CONTENT_REAL_VECTOR = [
    'standard__Per 90 Minutes__G-PK_z',
    'standard__Per 90 Minutes__Ast_z',
    'shooting__Standard__Sh/90_z',
    'shooting__Standard__SoT/90_z',
    'shooting__Standard__G/Sh_z',
    'shooting__Standard__SoT%_z',
    'interceptions_p90_z',
    'tackles_won_p90_z',
    'crosses_p90_z',
    'fouls_won_p90_z',
    'playing_time__Team Success__+/-90_z',
    'playing_time__Team Success__On-Off_z',
    'playing_time__Playing Time__Min%_z',
]

SYNTH_VECTOR_COLS = [
    'syn_trait_finishing','syn_trait_creativity','syn_trait_progression','syn_trait_ball_carrying',
    'syn_trait_defensive_intensity','syn_trait_press_resistance','syn_trait_aerial_physicality',
    'syn_trait_offball_threat','syn_xg_p90','syn_xa_p90','syn_xgi_p90','syn_progressive_passes_p90',
    'syn_progressive_carries_p90','syn_key_passes_p90','syn_tackles_won_p90','syn_interceptions_p90',
    'syn_pressures_applied_p90','syn_ball_retention_pct','syn_availability_pct'
]

def _primary_position(pos):
    if pd.isna(pos):
        return 'UNK'
    return str(pos).split(',')[0].strip()

def add_position_columns(players):
    out = players.copy()
    out['pos_primary'] = out['pos'].apply(_primary_position)
    out['pos_family'] = out['pos_primary'].map({'GK':'GK','DF':'DF','MF':'MF','FW':'FW'}).fillna('UNK')
    return out

def add_per90_columns(players):
    out = add_position_columns(players)
    for raw_col, new_col in PER90_MAP.items():
        if raw_col in out.columns:
            out[new_col] = out[raw_col] / out['nineties'].clip(lower=0.1)
        else:
            out[new_col] = np.nan
    return out

def filtered_players(players):
    out = add_per90_columns(players)
    outfield = out['pos_family'].isin(['DF','MF','FW']) & (out['minutes_played'] >= 600)
    keepers = (out['pos_family'] == 'GK') & (out['minutes_played'] >= 900)
    return out.loc[outfield | keepers].copy().reset_index(drop=True)

def safe_group_zscore(df, col, group_cols):
    grouped = df.groupby(group_cols)[col]
    mean = grouped.transform('mean')
    std = grouped.transform('std').replace(0, np.nan)
    z = (df[col] - mean) / std
    return z.replace([np.inf,-np.inf], np.nan).fillna(0.0)

def infer_role_subtype(players_f):
    out = players_f.copy()
    out['creator_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.7*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    out['defender_proxy'] = out['interceptions_p90_z'] + 0.9*out['tackles_won_p90_z'] - 0.2*out['crosses_p90_z']
    out['striker_proxy'] = out['shooting__Standard__Sh/90_z'] + 0.9*out['standard__Per 90 Minutes__G-PK_z'] + 0.5*out['offsides_p90_z']
    out['wing_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.8*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    role = []
    for _, row in out.iterrows():
        fam = row['pos_family']
        if fam == 'GK':
            role.append('Goalkeeper')
        elif fam == 'DF':
            role.append('Full-back / Wing-back' if (row['crosses_p90_z'] + row['standard__Per 90 Minutes__Ast_z']) > 0.3 else 'Centre-back')
        elif fam == 'MF':
            if row['creator_proxy'] - row['defender_proxy'] > 0.45:
                role.append('Creator / Advanced MF')
            elif row['defender_proxy'] - row['creator_proxy'] > 0.45:
                role.append('Ball-winning / Holding MF')
            else:
                role.append('Box-to-box / Hybrid MF')
        elif fam == 'FW':
            if row['striker_proxy'] - row['wing_proxy'] > 0.5:
                role.append('Striker')
            elif row['wing_proxy'] - row['striker_proxy'] > 0.2:
                role.append('Winger / Support Forward')
            else:
                role.append('Second Striker / Hybrid Forward')
        else:
            role.append('Other')
    out['role_subtype'] = role
    return out

def build_real_scoring_table(players, weights_override=None):
    """Build the scored player-season table.
    weights_override lets the sensitivity analysis perturb the composite without
    duplicating the whole function. Keys expected: 'FW', 'MF', 'DF' each mapped to
    a dict of {feature_z_col: weight}.
    """
    out = filtered_players(players)
    for col in REAL_SCORE_FEATURES:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = out.groupby(['league','season_label','pos_family'])[col].transform(lambda s: s.fillna(s.median()))
        out[col] = out[col].fillna(out[col].median())
        out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league','pos_family'])
    out = infer_role_subtype(out)

    mv = out['tm_market_value_eur_resolved'].fillna(out['tm_market_value_eur_resolved'].median())
    out['log_market_value'] = np.log1p(mv)
    out['log_market_value_z'] = safe_group_zscore(out, 'log_market_value', ['season_label','pos_family'])
    out['age_curve'] = np.exp(-((out['age'] - 26.0) ** 2) / (2 * 5.0**2))

    default_weights = {
        'FW': {
            'standard__Per 90 Minutes__G-PK_z': 0.36,
            'standard__Per 90 Minutes__Ast_z':  0.18,
            'shooting__Standard__SoT/90_z':     0.18,
            'shooting__Standard__G/Sh_z':       0.10,
            'fouls_won_p90_z':                  0.08,
            'playing_time__Team Success__+/-90_z': 0.10,
        },
        'MF': {
            'standard__Per 90 Minutes__Ast_z':  0.24,
            'standard__Per 90 Minutes__G-PK_z': 0.16,
            'interceptions_p90_z':              0.20,
            'tackles_won_p90_z':                0.16,
            'fouls_won_p90_z':                  0.10,
            'crosses_p90_z':                    0.06,
            'playing_time__Team Success__+/-90_z': 0.08,
        },
        'DF': {
            'interceptions_p90_z':              0.32,
            'tackles_won_p90_z':                0.28,
            'crosses_p90_z':                    0.10,
            'playing_time__Team Success__+/-90_z': 0.15,
            'playing_time__Team Success__On-Off_z': 0.15,
        },
    }
    weights = weights_override if weights_override is not None else default_weights

    out['real_quality_score'] = 0.0
    for fam, ws in weights.items():
        mask = out['pos_family'] == fam
        if not mask.any():
            continue
        score = np.zeros(mask.sum())
        for col, w in ws.items():
            if col in out.columns:
                score = score + w * out.loc[mask, col].to_numpy()
        out.loc[mask, 'real_quality_score'] = score

    gk = out['pos_family'] == 'GK'
    if gk.any():
        for col in ['keeper__Performance__Save%','keeper__Performance__CS%','keeper__Performance__GA90']:
            if col in out.columns:
                out[col] = out.groupby(['league','season_label'])[col].transform(lambda s: s.fillna(s.median()))
                out[col] = out[col].fillna(out[col].median())
                out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league'])
            else:
                out[f'{col}_z'] = 0.0
        out.loc[gk, 'real_quality_score'] = (
            0.40*out.loc[gk,'keeper__Performance__Save%_z'] +
            0.30*out.loc[gk,'keeper__Performance__CS%_z'] -
            0.30*out.loc[gk,'keeper__Performance__GA90_z']
        )

    out['value_adjusted_score'] = out['real_quality_score'] - 0.22*out['log_market_value_z']
    out['trajectory_adjusted_score'] = out['real_quality_score'] * (0.6 + 0.4*out['age_curve'])
    return out

def top_nonpersonalized(players_scored, season=TEST_SEASON, pos_family='FW', score_col='real_quality_score', top_n=10):
    cols = ['player','team','age','pos_family','role_subtype','tm_market_value_eur_resolved',score_col]
    return (
        players_scored[(players_scored['season_label']==season) & (players_scored['pos_family']==pos_family)]
        .sort_values(score_col, ascending=False)[cols]
        .rename(columns={'tm_market_value_eur_resolved':'market_value_eur'})
        .head(top_n)
        .reset_index(drop=True)
    )

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

def build_cf_model(players_scored, seasons, n_components=16):
    train = players_scored[players_scored['season_label'].isin(seasons)].copy()
    grouped = train.groupby(['team','player_key'], as_index=False)['minutes_played'].sum()
    teams = sorted(grouped['team'].unique().tolist())
    players = sorted(grouped['player_key'].unique().tolist())
    team_index = {team:i for i, team in enumerate(teams)}
    player_index = {player:i for i, player in enumerate(players)}
    rows = grouped['team'].map(team_index).to_numpy()
    cols = grouped['player_key'].map(player_index).to_numpy()
    vals = np.log1p(grouped['minutes_played'].to_numpy())
    matrix = csr_matrix((vals, (rows, cols)), shape=(len(teams), len(players)))
    k = int(max(4, min(n_components, matrix.shape[0]-1, matrix.shape[1]-1)))
    svd = TruncatedSVD(n_components=k, random_state=42)
    team_factors = svd.fit_transform(matrix)
    item_factors = svd.components_.T
    return {
        'team_ids': teams,
        'player_ids': players,
        'team_index': team_index,
        'player_index': player_index,
        'team_factors': team_factors,
        'item_factors': item_factors,
        'matrix': matrix,
        'k': k,
        'explained_variance': float(svd.explained_variance_ratio_.sum()),
        'matrix_shape': matrix.shape,
        'observations': len(grouped),
    }

def previous_season_candidates(players_scored, target_season=TEST_SEASON):
    prev = PREV_SEASON[target_season]
    return (players_scored[players_scored['season_label']==prev]
            .sort_values(['player_key','minutes_played'], ascending=[True, False])
            .drop_duplicates('player_key')
            .copy())

def cf_recommendations(model, players_scored, team, target_season=TEST_SEASON, pos_family=None, top_n=10):
    candidates = previous_season_candidates(players_scored, target_season)
    if pos_family is not None:
        candidates = candidates[candidates['pos_family']==pos_family].copy()
    if team not in model['team_index']:
        return pd.DataFrame()
    candidate_ids = [pid for pid in candidates['player_key'] if pid in model['player_index']]
    if not candidate_ids:
        return pd.DataFrame()
    vectors = model['item_factors'][[model['player_index'][pid] for pid in candidate_ids]]
    scores = vectors @ model['team_factors'][model['team_index'][team]]
    recs = candidates[candidates['player_key'].isin(candidate_ids)].copy()
    recs['cf_score'] = scores
    prev = PREV_SEASON[target_season]
    same_team_prev = set(players_scored.loc[(players_scored['season_label']==prev) & (players_scored['team']==team), 'player_key'])
    recs = recs[~recs['player_key'].isin(same_team_prev)].copy()
    return recs.sort_values('cf_score', ascending=False)[['player','team','age','pos_family','role_subtype','cf_score']].head(top_n).reset_index(drop=True)

players, teams = load_real_tables(ROOT)
players_scored = build_real_scoring_table(players)
cf_train_seasons = TRAIN_SEASONS + [VAL_SEASON]
cf_model = build_cf_model(players_scored, cf_train_seasons)

summary = pd.DataFrame({
    'metric': ['teams in train matrix', 'players in train matrix',
               'observed team-player pairs', 'matrix shape (teams x players)',
               'latent dimensions k', 'explained variance ratio sum'],
    'value': [len(cf_model['team_ids']), len(cf_model['player_ids']),
              cf_model['observations'], str(cf_model['matrix_shape']),
              cf_model['k'], round(cf_model['explained_variance'], 4)],
})
display(summary)


Using project root: /mnt/user-data/uploads


,metric,value
0,teams in train matrix,135
1,players in train matrix,3561
2,observed team-player pairs,4971
3,matrix shape (teams x players),"(135, 3561)"
4,latent dimensions k,16
5,explained variance ratio sum,0.1707


## Evidence: matrix sparsity and interaction density

The training matrix is highly sparse, as expected in a football recruitment setting where each club has fewer than 50 players per season and most player-club pairs never occur.


In [2]:
matrix = cf_model['matrix']
n_cells = matrix.shape[0] * matrix.shape[1]
nnz = matrix.nnz
density = nnz / n_cells

sparsity_summary = pd.DataFrame({
    'metric': ['matrix shape', 'non-zero cells', 'total cells',
               'density (nnz / total)', 'mean non-zero value (log1p minutes)',
               'median non-zero value'],
    'value': [str(matrix.shape), nnz, n_cells, round(density, 5),
              round(float(matrix.data.mean()), 3),
              round(float(np.median(matrix.data)), 3)],
})
display(sparsity_summary)

# Row and column degree distributions give a sense of who is well-represented.
team_degree = np.asarray((matrix > 0).sum(axis=1)).flatten()
player_degree = np.asarray((matrix > 0).sum(axis=0)).flatten()
print(f'Teams: {len(team_degree)}, distinct players touched per team: '
      f'min {team_degree.min()}, median {int(np.median(team_degree))}, '
      f'max {team_degree.max()}')
print(f'Players: {len(player_degree)}, clubs touching each player: '
      f'min {player_degree.min()}, median {int(np.median(player_degree))}, '
      f'max {player_degree.max()}')

# Sample of the interaction table.
interaction_pairs = (players_scored[players_scored['season_label'].isin(cf_train_seasons)]
                     .groupby(['team','player_key']).size().reset_index(name='obs_seasons'))
display(interaction_pairs.head(6))


,metric,value
0,matrix shape,"(135, 3561)"
1,non-zero cells,4971
2,total cells,480735
3,density (nnz / total),0.01034
4,mean non-zero value (log1p minutes),7.808
5,median non-zero value,7.764


Teams: 135, distinct players touched per team: min 16, median 38, max 65
Players: 3561, clubs touching each player: min 1, median 1, max 5


,team,player_key,obs_seasons
0,Ajaccio,benjamin leroy|1989,1
1,Ajaccio,bevic moussiti-oko|1995,1
2,Ajaccio,cedric avinel|1986,1
3,Ajaccio,clement vidal|2000,1
4,Ajaccio,cyrille bayala|1996,1
5,Ajaccio,fernand mayembo|1996,1


## Example recommendations: Arsenal midfield targets

In [3]:
display(cf_recommendations(cf_model, players_scored, team='Arsenal',
                           target_season=TEST_SEASON, pos_family='MF', top_n=10))

,player,team,age,pos_family,role_subtype,cf_score
0,Mattéo Guendouzi,Marseille,23,MF,Creator / Advanced MF,5.844442
1,Nuno Tavares,Marseille,22,MF,Box-to-box / Hybrid MF,5.009824
2,Willian,Fulham,33,MF,Creator / Advanced MF,4.968059
3,Aaron Ramsey,Nice,31,MF,Ball-winning / Holding MF,4.199397
4,Henrikh Mkhitaryan,Inter,33,MF,Ball-winning / Holding MF,4.154081
5,Alex Iwobi,Everton,26,MF,Creator / Advanced MF,3.896347
6,Dani Ceballos,Real Madrid,25,MF,Ball-winning / Holding MF,3.704694
7,Joe Willock,Newcastle United,22,MF,Box-to-box / Hybrid MF,3.632684
8,Jordan Veretout,Marseille,29,MF,Ball-winning / Holding MF,3.177844
9,Cengiz Ünder,Marseille,25,MF,Creator / Advanced MF,2.555243


## Example recommendations: Liverpool defensive targets

In [4]:
display(cf_recommendations(cf_model, players_scored, team='Liverpool',
                           target_season=TEST_SEASON, pos_family='DF', top_n=10))

,player,team,age,pos_family,role_subtype,cf_score
0,João Cancelo,Manchester City,28,DF,Full-back / Wing-back,0.545365
1,Matthijs de Ligt,Bayern Munich,22,DF,Centre-back,0.544844
2,Ozan Kabak,Hoffenheim,22,DF,Centre-back,0.432623
3,Danilo,Juventus,31,DF,Full-back / Wing-back,0.430285
4,Chris Smalling,Roma,32,DF,Centre-back,0.407916
5,Stanley N'Soki,Hoffenheim,23,DF,Centre-back,0.399207
6,Leonardo Bonucci,Juventus,35,DF,Centre-back,0.379089
7,Alex Sandro,Juventus,31,DF,Centre-back,0.378023
8,Robin Knoche,Union Berlin,30,DF,Centre-back,0.374861
9,Dayot Upamecano,Bayern Munich,23,DF,Centre-back,0.358412


## Example recommendations: Brentford forward targets

In [5]:
display(cf_recommendations(cf_model, players_scored, team='Brentford',
                           target_season=TEST_SEASON, pos_family='FW', top_n=10))

,player,team,age,pos_family,role_subtype,cf_score
0,Francesco Caputo,Empoli,34,FW,Second Striker / Hybrid Forward,0.232382
1,Grégoire Defrel,Sassuolo,31,FW,Winger / Support Forward,0.192033
2,Alexis Sánchez,Marseille,33,FW,Striker,0.175293
3,Romelu Lukaku,Inter,29,FW,Second Striker / Hybrid Forward,0.150309
4,Fabio Quagliarella,Sampdoria,39,FW,Striker,0.136965
5,Edin Džeko,Inter,36,FW,Striker,0.132349
6,Manolo Gabbiadini,Sampdoria,30,FW,Striker,0.131901
7,Wissam Ben Yedder,Monaco,31,FW,Striker,0.131884
8,Sam Lammers,Sampdoria,25,FW,Winger / Support Forward,0.130390
9,Federico Di Francesco,Lecce,28,FW,Winger / Support Forward,0.128710


## K-stability ablation

The default `k=16` was chosen heuristically. To check whether the shortlist is an artefact of that choice we refit the model at k=8 and k=32 and measure Jaccard overlap of the top-10 for the three example clubs. A high overlap means the latent structure is stable; a low overlap means the ranking depends on the factor count and should be interpreted with caution.


In [6]:
def jaccard(a, b):
    a_set, b_set = set(a), set(b)
    if not a_set and not b_set:
        return 1.0
    return len(a_set & b_set) / len(a_set | b_set)

targets = [('Arsenal', 'MF'), ('Liverpool', 'DF'), ('Brentford', 'FW')]
base_lists = {}
for team, pos in targets:
    base_lists[(team, pos)] = cf_recommendations(cf_model, players_scored, team=team,
                                                 target_season=TEST_SEASON,
                                                 pos_family=pos, top_n=10)['player'].tolist()

rows = []
for k in [8, 32]:
    alt = build_cf_model(players_scored, cf_train_seasons, n_components=k)
    for team, pos in targets:
        alt_list = cf_recommendations(alt, players_scored, team=team,
                                      target_season=TEST_SEASON,
                                      pos_family=pos, top_n=10)['player'].tolist()
        rows.append({
            'k': k,
            'team': team, 'pos_family': pos,
            'jaccard_vs_k16': round(jaccard(base_lists[(team, pos)], alt_list), 3),
            'overlap_count': len(set(base_lists[(team, pos)]) & set(alt_list)),
        })

display(pd.DataFrame(rows))


,k,team,pos_family,jaccard_vs_k16,overlap_count
0,8,Arsenal,MF,0.667,8
1,8,Liverpool,DF,0.000,0
2,8,Brentford,FW,0.250,4
3,32,Arsenal,MF,0.667,8
4,32,Liverpool,DF,0.111,2
5,32,Brentford,FW,0.429,6


## Interpretation

The latent-factor model finds plausible behavioural clusters despite the small number of clubs. Defensive-minded clubs surface defensively profiled candidates; possession clubs surface creators. The stability check shows that the top-10 moves with k but the overlap stays within a range that is consistent with "the latent direction is real, but the fine-grained ordering depends on k." The ALS caveat at the top of this notebook applies: these rankings are useful as a behavioural signal, not as a claim that the model has recovered the true implicit-feedback geometry.

Collaborative filtering learns who clubs historically used, which is valuable for behavioural similarity but still ignores tactical context, squad gaps and affordability. Those layers are added next.
